# Exploratory Data Analysis Walkthrough

This notebook demonstrates the **notebook -> tested src extraction** workflow.
Every analytical step calls a function from the tested `src.eda` library; the
cells contain *no business logic* of their own. That is the whole point of this
exemplar: a notebook is the fast, interactive entry point a researcher reaches
for first, but the moment a computation matters it moves into `src/` where it is
typed, tested (>= 90% coverage), and reusable from scripts and the manuscript.

Run order is top-to-bottom. The notebook is also exercised structurally by the
test suite, so it cannot silently drift from the library it imports.

## 1. Make the library importable

The project ships its EDA logic under `src/`. We add the project root to the
path so `from src import ...` resolves whether the notebook is launched from the
project directory or the monorepo root.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").is_dir() is False and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    clean_dataset,
    correlation_matrix,
    correlation_heatmap_data,
    group_count_data,
    group_means,
    histogram_data,
    load_dataset,
    normalize_numeric,
    strongest_pairs,
    summary_statistics,
)

## 2. Load the dataset

`load_dataset()` reads the shipped, deterministic CSV fixture
(`data/measurements.csv`) and coerces numeric columns, preserving missing cells
as `NaN` rather than silently imputing them.

In [ ]:
raw = load_dataset()
print("shape:", raw.shape)
print("missing per column:\n", raw.isna().sum())
raw.head()

## 3. Clean: drop incomplete rows (explicitly reported)

`clean_dataset()` removes rows missing any numeric feature and returns a report
so the missingness is surfaced, not hidden.

In [ ]:
clean, report = clean_dataset(raw)
print(report)

## 4. Summary statistics and per-group means

In [ ]:
for stat in summary_statistics(clean):
    print(stat)

group_means(clean)

## 5. Correlation structure

`correlation_matrix()` plus `strongest_pairs()` rank the most strongly related
features — usually the single most useful artifact of a first EDA pass.

In [ ]:
matrix = correlation_matrix(clean)
print("strongest pairs:")
for col_a, col_b, value in strongest_pairs(matrix, top_n=3):
    print(f"  {col_a} ~ {col_b}: r = {value:+.3f}")
matrix

## 6. Figure-data preparers feed plotting

The library returns *plot-ready data structures*; matplotlib lives only in the
notebook / scripts, never in `src/`. This keeps the library importable on a
headless machine and the figure preparers trivially testable.

In [ ]:
import matplotlib.pyplot as plt

hist = histogram_data(clean, "height_cm", bins=10)
widths = [b - a for a, b in zip(hist.edges, hist.edges[1:])]
plt.bar(hist.edges[:-1], hist.counts, width=widths, align="edge")
plt.title("Height distribution")
plt.xlabel("height_cm")
plt.ylabel("count")
plt.show()

In [ ]:
heat = correlation_heatmap_data(clean)
plt.imshow(heat.values, vmin=-1.0, vmax=1.0, cmap="coolwarm")
plt.xticks(range(len(heat.labels)), heat.labels, rotation=45, ha="right")
plt.yticks(range(len(heat.labels)), heat.labels)
plt.title("Feature correlation")
plt.colorbar()
plt.show()

In [ ]:
groups = group_count_data(clean)
plt.bar(groups.labels, groups.counts)
plt.title("Rows per group")
plt.show()

## 7. Normalize for cross-feature comparison

`normalize_numeric()` z-score standardizes the numeric columns so features on
different scales (cm vs kg vs bpm) become directly comparable.

In [ ]:
norm = normalize_numeric(clean)
norm[["height_cm", "weight_kg", "resting_hr_bpm"]].describe().loc[["mean", "std"]]

## Where to go next

When a cell here grows beyond a one-line call, extract it into `src/eda/` with a
test first (TDD), then call the new function from the notebook. The thin script
`scripts/eda_analysis.py` already runs this exact pipeline headless and writes
the figures + summary table into `output/`.